# M16 • Memory-Systeme

> **Kurs:** KI-Agenten. Verstehen. Anwenden. Gestalten.
> **Modul:** M16 – Kurz- und Langzeitgedächtnis für persistente KI-Agenten

## Lernziele

Nach diesem Modul kannst du:

- **Kurzzeit-Memory** einsetzen: Conversation Buffer, Sliding Window, Summarization
- **Langzeit-Memory** aufbauen: Vektordatenbank und Entity Memory
- **Per-User-Memory** mit Thread-IDs realisieren
- Verschiedene Memory-Strategien kombinieren

---

## Warum brauchen Agenten Memory?

Ein LLM hat kein eingebautes Gedächtnis. Ohne explizite Mechanismen beginnt jedes Gespräch von vorne.

| Problem | Lösung |
|---------|--------|
| Gesprächsverlauf geht verloren | Kurzzeit-Memory im State |
| Kontext wächst unbegrenzt | Sliding Window oder Summarization |
| Nutzerdaten nach Session weg | Langzeit-Memory (Vektordatenbank) |
| Kein Personalisierungspotenzial | Entity Memory oder Per-User-Store |

In [ ]:
import os
import operator
from typing import TypedDict, Annotated, Optional

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.messages import trim_messages, RemoveMessage, SystemMessage, HumanMessage, AIMessage

try:
    from genai_lib.utilities import mprint, mermaid, setup_api_keys
    setup_api_keys(["OPENAI_API_KEY"])
except ImportError:
    from IPython.display import display, Markdown
    def mprint(t): display(Markdown(t))
    def mermaid(d): print(d)

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
mprint("✅ Setup abgeschlossen")

In [ ]:
mermaid('''
flowchart TB
    M[Memory-Systeme] --> K[Kurzzeit-Memory]
    M --> L[Langzeit-Memory]

    K --> K1[Conversation Buffer\nVoller Verlauf]
    K --> K2[Sliding Window\nLetzte N Nachrichten]
    K --> K3[Summarization\nKomprimierter Verlauf]

    L --> L1[Semantisches Memory\nVektordatenbank]
    L --> L2[Entity Memory\nKey-Value Struktur]
''')

---
## 1 Kurzzeit-Memory

Drei Strategien, die sich darin unterscheiden, wie viel Verlauf aufbewahrt wird.

### 1.1 Conversation Buffer – Vollständiger Verlauf

In [ ]:
class BufferState(TypedDict):
    messages: Annotated[list, add_messages]

def buffer_chat(state: BufferState) -> BufferState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(BufferState)
graph.add_node("chat", buffer_chat)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)

buffer_app = graph.compile(checkpointer=MemorySaver())
mprint("✅ Conversation Buffer aufgebaut")

In [ ]:
config = {"configurable": {"thread_id": "buffer-demo"}}

for nachricht in ["Mein Name ist Emma.", "Ich arbeite als Datenwissenschaftlerin.", "Wie heiße ich?"]:
    result = buffer_app.invoke(
        {"messages": [HumanMessage(content=nachricht)]},
        config=config
    )
    antwort = result["messages"][-1].content
    mprint(f"**Nutzer:** {nachricht}  \n**Agent:** {antwort}")
    mprint("---")

### 1.2 Sliding Window – Letzte N Nachrichten

`trim_messages()` begrenzt den Kontext auf die jüngsten Nachrichten, sobald ein Token-Limit erreicht wird.

In [ ]:
class WindowState(TypedDict):
    messages: Annotated[list, add_messages]

def window_chat(state: WindowState) -> WindowState:
    trimmed = trim_messages(
        state["messages"],
        max_tokens=2000,
        strategy="last",
        token_counter=llm,
        include_system=True,
        allow_partial=False,
    )
    response = llm.invoke(trimmed)
    return {"messages": [response]}

graph2 = StateGraph(WindowState)
graph2.add_node("chat", window_chat)
graph2.add_edge(START, "chat")
graph2.add_edge("chat", END)
window_app = graph2.compile(checkpointer=MemorySaver())

# Veranschaulichung: trim_messages kuerzt
nachrichten = [HumanMessage(content=f"Nachricht {i}") for i in range(1, 8)]
gekuerzt = trim_messages(nachrichten, max_tokens=100, strategy="last", token_counter=llm)
mprint(f"Original: {len(nachrichten)} Nachrichten → Nach trim_messages: {len(gekuerzt)} Nachrichten")

### 1.3 Summarization Memory – Komprimierter Verlauf

Ältere Nachrichten werden vom LLM zusammengefasst statt verworfen. Der Kontext bleibt erhalten, der Token-Verbrauch wird begrenzt.

In [ ]:
class SummaryState(TypedDict):
    messages: Annotated[list, add_messages]
    summary: str

def summarize_if_needed(state: SummaryState) -> SummaryState:
    # Komprimiert, sobald mehr als 6 Nachrichten vorhanden sind
    messages = state["messages"]
    if len(messages) < 6:
        return {}

    existing = state.get("summary", "")
    behalten = messages[-4:]
    zum_komprimieren = messages[:-4]

    prompt = (
        f"Fasse dieses Gespräch in 2-3 Sätzen zusammen.\n"
        f"Bisherige Zusammenfassung: {existing or 'Keine'}\n\n"
        + "\n".join(f"{m.type}: {m.content}" for m in zum_komprimieren)
    )
    neue_zusammenfassung = llm.invoke(prompt).content
    zu_entfernen = [RemoveMessage(id=m.id) for m in zum_komprimieren]
    summary_msg = SystemMessage(
        content=f"Bisheriger Verlauf (komprimiert): {neue_zusammenfassung}"
    )
    return {"messages": [summary_msg] + zu_entfernen, "summary": neue_zusammenfassung}

def summary_chat(state: SummaryState) -> SummaryState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph3 = StateGraph(SummaryState)
graph3.add_node("summarize", summarize_if_needed)
graph3.add_node("chat", summary_chat)
graph3.add_edge(START, "summarize")
graph3.add_edge("summarize", "chat")
graph3.add_edge("chat", END)
summary_app = graph3.compile(checkpointer=MemorySaver())
mprint("✅ Summarization Memory aufgebaut")

In [ ]:
config = {"configurable": {"thread_id": "summary-demo"}}
themen = [
    "Mein Name ist Max, ich bin Softwareentwickler.",
    "Ich arbeite bei einer Berliner KI-Firma.",
    "Mein Lieblingswerkzeug ist Python.",
    "Ich interessiere mich für LangChain und LangGraph.",
    "Was weißt du über mich?",
]
for nachricht in themen:
    result = summary_app.invoke(
        {"messages": [HumanMessage(content=nachricht)]},
        config=config
    )
    antwort = result["messages"][-1].content
    n = len(result["messages"])
    mprint(f"**({n} Msgs) Nutzer:** {nachricht}  \n**Agent:** {antwort[:140]}...")
    mprint("---")

---
## 2 Langzeit-Memory

Langzeit-Memory überlebt das Sitzungsende und steht in zukünftigen Gesprächen zur Verfügung.

### 2.1 Semantisches Memory – Vektordatenbank

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
memory_store = Chroma(
    collection_name="agent_memory",
    embedding_function=embeddings,
)

@tool
def memory_speichern(information: str) -> str:
    '''MEMORY SPEICHERN - Speichert eine wichtige Information dauerhaft.

    Verwende dieses Tool wenn der Nutzer Relevantes erwähnt:
    - Persönliche Fakten ("Ich arbeite als Arzt")
    - Präferenzen ("Ich mag kurze Antworten")
    - Ziele ("Ich lerne Python fuer Data Science")

    Args:
        information: Die Information in einem präzisen Satz

    Returns:
        Bestätigung der Speicherung
    '''
    memory_store.add_texts([information])
    return f"Gespeichert: {information}"

@tool
def memory_abrufen(frage: str) -> str:
    '''MEMORY ABRUFEN - Durchsucht das Gedächtnis nach relevanten Informationen.

    Args:
        frage: Suchanfrage in natürlicher Sprache

    Returns:
        Relevante gespeicherte Informationen
    '''
    docs = memory_store.similarity_search(frage, k=3)
    if not docs:
        return "Keine relevanten Informationen gefunden."
    return "\n".join(f"- {d.page_content}" for d in docs)

mprint("✅ Memory-Tools definiert")

In [ ]:
from langchain.agents import create_agent

memory_agent = create_agent(
    model=llm,
    tools=[memory_speichern, memory_abrufen],
    system_prompt=(
        "Du bist ein hilfreicher Assistent mit Langzeitgedächtnis. "
        "Speichere wichtige Informationen über den Nutzer und rufe sie bei Bedarf ab."
    )
)

result1 = memory_agent.invoke({
    "messages": [HumanMessage(content="Ich heiße Lena und lerne KI-Programmierung.")]
})
mprint(f"**Agent:** {result1['messages'][-1].content}")

result2 = memory_agent.invoke({
    "messages": [HumanMessage(content="Was weißt du über mich?")]
})
mprint(f"**Agent:** {result2['messages'][-1].content}")

### 2.2 Entity Memory – Key-Value für Entitäten

In [ ]:
from pydantic import BaseModel, Field

class Entitaet(BaseModel):
    name: str = Field(description="Name der Entität (Person, Ort, Projekt)")
    beschreibung: str = Field(description="Kurze Beschreibung in einem Satz")

class EntitaetListe(BaseModel):
    entitaeten: list[Entitaet] = Field(default_factory=list)

class EntityState(TypedDict):
    messages: Annotated[list, add_messages]
    entity_memory: dict

extractor = llm.with_structured_output(EntitaetListe)

def entity_extraktor(state: EntityState) -> EntityState:
    # Extrahiert Entitaeten aus der letzten Nutzernachricht
    letzte = state["messages"][-1].content
    result = extractor.invoke(
        f"Extrahiere wichtige Entitäten (Personen, Projekte, Orte) aus:\n{letzte}"
    )
    updated = dict(state.get("entity_memory", {}))
    for e in result.entitaeten:
        updated[e.name] = e.beschreibung
    return {"entity_memory": updated}

def entity_chat(state: EntityState) -> EntityState:
    kontext = "\n".join(f"- {k}: {v}" for k, v in state.get("entity_memory", {}).items())
    system = f"Du kennst folgende Entitäten:\n{kontext}" if kontext else ""
    msgs = ([SystemMessage(content=system)] if system else []) + list(state["messages"])
    return {"messages": [llm.invoke(msgs)]}

graph4 = StateGraph(EntityState)
graph4.add_node("extraktor", entity_extraktor)
graph4.add_node("chat", entity_chat)
graph4.add_edge(START, "extraktor")
graph4.add_edge("extraktor", "chat")
graph4.add_edge("chat", END)
entity_app = graph4.compile(checkpointer=MemorySaver())
mprint("✅ Entity Memory aufgebaut")

In [ ]:
config = {"configurable": {"thread_id": "entity-demo"}}
nachrichten = [
    "Ich arbeite mit Tom an Projekt Alpha.",
    "Tom ist unser Tech Lead und Projekt Alpha ist eine KI-Plattform.",
    "Was weißt du über Tom?",
]
for msg in nachrichten:
    result = entity_app.invoke(
        {"messages": [HumanMessage(content=msg)]},
        config=config
    )
    mem = result.get("entity_memory", {})
    antwort = result["messages"][-1].content
    mprint(f"**Nutzer:** {msg}")
    if mem:
        mprint(f"*Entity Memory: {mem}*")
    mprint(f"**Agent:** {antwort}")
    mprint("---")

---
## 3 Per-User Memory

In Multi-User-Systemen trennt eine eindeutige `thread_id` pro Nutzer den Kontext vollständig.

In [ ]:
def get_config(user_id: str, session_id: str = "default") -> dict:
    # Erstellt nutzerspezifische Konfiguration
    return {"configurable": {"thread_id": f"{user_id}:{session_id}"}}

config_alice = get_config("alice")
config_bob   = get_config("bob")

buffer_app.invoke(
    {"messages": [HumanMessage(content="Ich heiße Alice und mag Python.")]},
    config=config_alice
)
buffer_app.invoke(
    {"messages": [HumanMessage(content="Ich heiße Bob und arbeite mit Java.")]},
    config=config_bob
)

# Beide fragen nach ihrem Namen - vollstaendig getrennte Kontexte
for name, cfg in [("Alice", config_alice), ("Bob", config_bob)]:
    result = buffer_app.invoke(
        {"messages": [HumanMessage(content="Wie heiße ich?")]},
        config=cfg
    )
    mprint(f"**{name} fragt:** Wie heiße ich?  \n**Agent:** {result['messages'][-1].content}")

---
## Zusammenfassung

| Memory-Typ | Implementierung | Einsatz |
|-----------|----------------|---------|
| **Conversation Buffer** | `add_messages` Reducer | Kurze Gespräche, einfache Demos |
| **Sliding Window** | `trim_messages()` | Lange Gespräche, Token-Budget begrenzen |
| **Summarization** | `RemoveMessage` + LLM | Multi-Turn mit wichtigem Kontext |
| **Semantisches Memory** | ChromaDB + `@tool` | Nutzerpräferenzen über Sessions |
| **Entity Memory** | `with_structured_output` + Dict | CRM-Agenten, Support-Systeme |
| **Per-User Memory** | Thread-IDs im Checkpointer | Multi-User-Anwendungen |

**Faustregel:**
- **Kurzzeit immer** → Conversation Buffer oder Sliding Window
- **Langzeit bei Personalisierung** → Semantisches oder Entity Memory
- **Per-User** → eindeutige Thread-IDs + persistenter Checkpointer

Verwandte Module: M15 (Checkpointing), M18 (Human-in-the-Loop), M19 (Multi-Agent)